In [25]:
import xml.etree.ElementTree as ET
import pandas as pd
import itertools
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from difflib import SequenceMatcher
from collections import defaultdict
mari_term_file = 'C:/Users/lucia/OneDrive/Desktop/Tirocinio/XML/MariT.xml'
ital_wn_file = 'C:/Users/lucia/OneDrive/Desktop/Tirocinio/XML/IWN.xml'
candidates_csv = 'C:/Users/lucia/OneDrive/Desktop/shared_lemmas.csv'
transcription_csv = 'C:/Users/lucia/OneDrive/Desktop/results_transcribed.csv'

In [26]:
def parse_xml(file_path):
    return ET.parse(file_path).getroot()

def extract_word_meanings(xml_root):
    word_meanings = []
    for word_meaning in xml_root.findall(".//WORD_MEANING"):
        lemma_variants = word_meaning.find("VARIANTS")
        lemmas = [literal.get("LEMMA") for literal in lemma_variants.findall("LITERAL") if literal.get("LEMMA")]
        senses = [literal.get("SENSE") for literal in lemma_variants.findall("LITERAL") if literal.get("SENSE")]
        gloss = word_meaning.find("GLOSS").text or ""
        relations = []
        for relation in word_meaning.findall(".//INTERNAL_LINKS/RELATION"):
            target_wm = relation.find("TARGET_WM")
            if target_wm is not None:
                target_id = target_wm.get("ID")
                target_lemma = target_wm.get("LEMMA")
                target_gloss = target_wm.get("GLOSS", "No Gloss")
                relation_type = relation.get("TYPE")
                if relation_type in {"has_hyperonym", "has_hyponym", "near_synonym"}:
                    relations.append({
                        "relation_type": relation_type,
                        "target_id": target_id,
                        "target_lemma": target_lemma,
                        "target_gloss": target_gloss
                    })
        first_literal = lemma_variants.find("LITERAL")
        first_literal_lemma = first_literal.get("LEMMA") if first_literal is not None else "No Literal Lemma"
        
        word_meanings.append({
            "id": word_meaning.get("ID"),
            "lemmas": lemmas,
            "senses": senses,
            "gloss": gloss,
            "relations": relations,
            "first_literal_lemma": first_literal_lemma
        })
    return word_meanings

def calculate_gloss_similarity(gloss1, gloss2):
    if not gloss1.strip() or not gloss2.strip():
        return 0.0
    
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform([gloss1, gloss2])
    return cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:2])[0][0]

def calculate_relation_similarity(relations1, relations2, target_gloss_similarity):
    weights = {
        "has_hyperonym": 0.82,
        "has_hyponym": 0.76,
        "near_synonym": 0.6
    }
    
    relation_weights = {}
    total_weighted_similarity = 0
    bonus = 0
    malus = 0
    bonus_relations = []
    missing_relations_details = []
    no_gloss_relations = []
    no_gloss_words = []
    
    relations1_set = {(r["relation_type"], r["target_lemma"], r.get("target_sense")): r["target_gloss"] for r in relations1}
    relations2_dict = {(r["relation_type"], r["target_lemma"], r.get("target_sense")): r["target_gloss"] for r in relations2}
    
    for key, mari_gloss in relations1_set.items():
        if key in relations2_dict:
            ital_gloss = relations2_dict[key]
            gloss_similarity = calculate_gloss_similarity(mari_gloss, ital_gloss)
            # Bonus for matching glosses (both present or both "No Gloss")
            if (mari_gloss != "No Gloss" and ital_gloss != "No Gloss") or (mari_gloss == "No Gloss" and ital_gloss == "No Gloss"):
                bonus += 0.33
                bonus_relations.append(key[1])  # Store only the word
            # Malus for MariTerm gloss present but ItalWN gloss missing
            elif mari_gloss != "No Gloss" and ital_gloss == "No Gloss":
                malus -= 0.33
                no_gloss_relations.append(key[1])  # Store only the word
                no_gloss_words.append(mari_gloss)  # Add glosses with no gloss in ItalWN
            if mari_gloss.strip() and ital_gloss.strip():
                weight = weights.get(key[0], 0)
                weighted_similarity = gloss_similarity * weight
                relation_weights[key] = (weighted_similarity, gloss_similarity, weight)
                total_weighted_similarity += weighted_similarity
        else:
            # Malus for missing relations in ItalWN
            malus -= 0.33
            missing_relations_details.append(key[1])  # Store only the word
    
    total_similarity = total_weighted_similarity + bonus + malus
    malus_count = len(no_gloss_relations)  # Count of relations with no gloss
    return total_weighted_similarity, relation_weights, bonus, malus, malus_count, missing_relations_details, no_gloss_relations, no_gloss_words, total_similarity, bonus_relations

def format_relations(relations):
    formatted_relations = []
    for rel in relations:
        formatted_relations.append(f"{rel['relation_type']}, {rel['target_lemma']}, {rel['target_gloss']}")
    return formatted_relations

def match_lemmas_and_calculate_similarity(mari_term_wms, ital_wn_wms):
    matched_pairs = []
    
    for wm1, wm2 in itertools.product(mari_term_wms, ital_wn_wms):
        common_lemmas_and_senses = set(
            (l1, s1) for l1, s1 in zip(wm1["lemmas"], wm1["senses"])
        ) & set(
            (l2, s2) for l2, s2 in zip(wm2["lemmas"], wm2["senses"])
        )
        if common_lemmas_and_senses:
            wn_g_sim = calculate_gloss_similarity(wm1["gloss"], wm2["gloss"])
            
            total_weighted_similarity, relation_weights, bonus, malus, malus_count, missing_relations, no_gloss_relations, no_gloss_words, total_similarity, bonus_relations = calculate_relation_similarity(
                wm1["relations"], wm2["relations"], wn_g_sim
            )
            
            matched_pairs.append({
                "mari_term_id": wm1["id"],
                "ital_wn_id": wm2["id"],
                "mari_term_gloss": wm1["gloss"],
                "ital_wn_gloss": wm2["gloss"],
                "wn_g_sim": wn_g_sim,
                "total_weighted_similarity": total_weighted_similarity,
                "relation_weights": relation_weights,
                "bonus": bonus,
                "malus": malus,
                "malus_count": malus_count,
                "total_similarity": total_similarity,
                "mari_term_relations": format_relations(wm1["relations"]),
                "ital_wn_relations": format_relations(wm2["relations"]),
                "first_literal_lemma": wm1["first_literal_lemma"],
                "missing_relations": missing_relations,
                "no_gloss_relations": no_gloss_relations,
                "no_gloss_words": no_gloss_words,
                "bonus_relations": bonus_relations
            })
    
    return matched_pairs

def score_candidates(mari_term_file, ital_wn_file, candidates_csv):
    candidate_lemmas = set(pd.read_csv(candidates_csv, delimiter=';')['Shared Lemma'].str.lower())

    mari_term_wms = [wm for wm in extract_word_meanings(parse_xml(mari_term_file)) 
                     if any(lemma.lower() in candidate_lemmas for lemma in wm['lemmas'])]
    ital_wn_wms = [wm for wm in extract_word_meanings(parse_xml(ital_wn_file)) 
                    if any(lemma.lower() in candidate_lemmas for lemma in wm['lemmas'])]

    matched_pairs = match_lemmas_and_calculate_similarity(mari_term_wms, ital_wn_wms)
    matched_pairs.sort(key=lambda x: x["total_similarity"], reverse=True)

    for pair in matched_pairs:
        print(f"\n Literal Lemma: {pair['first_literal_lemma']}")
        print(f"  MariTerm Gloss: {pair['mari_term_gloss']}")
        print(f"  ItalWN Gloss: {pair['ital_wn_gloss']}")
        print(f"  MariTerm ID: {pair['mari_term_id']}")
        print(f"  ItalWN ID: {pair['ital_wn_id']}")
        print(f"  Gloss Similarity (Word Meaning): {pair['wn_g_sim']:.2f}")
        print(f"  Total Weighted Relation Similarity: {pair['total_weighted_similarity']:.2f}")
        print(f"  Bonus: {pair['bonus']:.2f} ({len(pair['bonus_relations'])}/{len(pair['mari_term_relations'])} - {', '.join(pair['bonus_relations'])})")
        print(f"  Malus: {pair['malus']:.2f} ({len(pair['missing_relations'])}/{len(pair['mari_term_relations'])} - No Gloss in ItalWN: {len(pair['no_gloss_relations'])}, {', '.join(pair['no_gloss_relations'])} - Missing: {', '.join(pair['missing_relations'])})")
        print(f"  Relation Similarity: {pair['wn_g_sim']:.2f} [Gloss Similarity: {pair['wn_g_sim']:.2f} * Weight]")
        print(f"  Total Similarity: {pair['total_similarity']:.2f}")
        print(f"  MariTerm Relations:")
        for rel in pair['mari_term_relations']:
            rel_type, target_lemma, target_gloss = rel.split(', ')
            if (rel_type, target_lemma) in pair['relation_weights']:
                weight_score, gloss_similarity, weight = pair['relation_weights'][(rel_type, target_lemma)]
                print(f"   {rel} (Score: {weight_score:.2f} = {gloss_similarity:.2f} * {weight})")
            else:
                print(f"   {rel} (Score: 0.00)")
        print(f"  ItalWN Relations:")
        for rel in pair['ital_wn_relations']:
            print(f"   {rel}")

score_candidates(mari_term_file, ital_wn_file, candidates_csv)


 Literal Lemma: nave militare
  MariTerm Gloss: nave impiegata per scopi militari e/o di guerra.
  ItalWN Gloss: 
  MariTerm ID: N#20
  ItalWN ID: N#49583
  Gloss Similarity (Word Meaning): 0.00
  Total Weighted Relation Similarity: 13.04
  Bonus: 5.94 (18/23 - nave, ammiraglia, cacciasommergibili, cacciatorpediniere, cannoniera, corazzata, corvetta, fregata, incrociatore, lanciamissili, posamine, posareti, serrafila, silurante, sommergibile, torpediniera, vascello, vedetta)
  Malus: -1.65 (5/23 - No Gloss in ItalWN: 0,  - Missing: barbotta, betta, dragamine, galea, portaerei)
  Relation Similarity: 0.00 [Gloss Similarity: 0.00 * Weight]
  Total Similarity: 17.33
  MariTerm Relations:
   has_hyperonym, nave, imbarcazione di una certa grandezza che  fornita di adeguati mezzi di propulsione  è adibita al trasporto di persone e merci o che  opportunamente attrezzata e armata  è impiegata in attività belliche.  (Score: 0.00)
   has_hyponym, ammiraglia, nave dell'ammiraglio (Score: 0.00)
 

In [27]:
def print_list(entry):
    """
    Print the details of a single entry in the specified format.
    """
    print(f"Literal Lemma: {entry['Literal Lemma']}")
    print(f"Mariterm ID: {entry['Mariterm ID']}")
    print(f"Total Similarity: {entry['Total Similarity']:.2f}")
    print(f"Mariterm Gloss: {entry['Mariterm Gloss']}")
    print("-" * 40)

def store_one_to_one(mari_term_file, ital_wn_file, candidates_csv):
    """
    Store 1-to-1 entries with Literal Lemma, MariTerm ID, Total Similarity, and MariTerm Gloss into a variable.
    """
    # Load the candidates lemmas
    candidate_lemmas = set(pd.read_csv(candidates_csv, delimiter=';')['Shared Lemma'].str.lower())

    # Extract word meanings from XML files
    mari_term_wms = extract_word_meanings(parse_xml(mari_term_file))
    ital_wn_wms = extract_word_meanings(parse_xml(ital_wn_file))

    # Filter word meanings based on candidate lemmas
    mari_term_wms = [wm for wm in mari_term_wms if any(lemma.lower() in candidate_lemmas for lemma in wm['lemmas'])]
    ital_wn_wms = [wm for wm in ital_wn_wms if any(lemma.lower() in candidate_lemmas for lemma in wm['lemmas'])]

    # Get the matched pairs
    matched_pairs = match_lemmas_and_calculate_similarity(mari_term_wms, ital_wn_wms)

    # Dictionary to track ID counts in MariTerm and ItalWN
    mari_term_id_to_ital_wn_ids = {}
    ital_wn_id_counts = {}

    for pair in matched_pairs:
        mari_term_id = pair['mari_term_id']
        ital_wn_id = pair['ital_wn_id']

        if mari_term_id not in mari_term_id_to_ital_wn_ids:
            mari_term_id_to_ital_wn_ids[mari_term_id] = [ital_wn_id]
        else:
            mari_term_id_to_ital_wn_ids[mari_term_id].append(ital_wn_id)

        if ital_wn_id in ital_wn_id_counts:
            ital_wn_id_counts[ital_wn_id] += 1
        else:
            ital_wn_id_counts[ital_wn_id] = 1

    # Filter matched pairs to find 1-to-1 entries
    unique_entries = []
    for pair in matched_pairs:
        mari_term_id = pair['mari_term_id']
        ital_wn_id = pair['ital_wn_id']

        # Check if MariTerm ID has exactly one corresponding ItalWN ID
        if len(mari_term_id_to_ital_wn_ids[mari_term_id]) == 1 and ital_wn_id_counts[ital_wn_id] == 1:
            entry = {
                "Literal Lemma": pair.get("first_literal_lemma", 'N/A'),
                "Mariterm ID": mari_term_id,
                "Total Similarity": pair.get("total_similarity", 0.00),
                "Mariterm Gloss": pair.get("mari_term_gloss", 'No Gloss Available')
            }
            unique_entries.append(entry)

    # Return the results as a list of dictionaries
    return unique_entries

unique_entries = store_one_to_one(mari_term_file, ital_wn_file, candidates_csv)

for entry in unique_entries:
    print_list(entry)


Literal Lemma: abbandonare
Mariterm ID: V#722
Total Similarity: 0.00
Mariterm Gloss: l'atto di lasciare la nave in pericolo da parte del capitano e dell'equipaggio.
----------------------------------------
Literal Lemma: abbattere
Mariterm ID: V#1871
Total Similarity: -0.66
Mariterm Gloss: far ruotare la nave alla fonda per mettere alla vela o riprendere il cammino.
----------------------------------------
Literal Lemma: abbeverare
Mariterm ID: V#2691
Total Similarity: 0.00
Mariterm Gloss: colmare d'acqua un'imbarcazione tirata in secco per far gonfiare il legname della chiglia cosicché collimi.
----------------------------------------
Literal Lemma: abbisciare
Mariterm ID: V#176
Total Similarity: 0.06
Mariterm Gloss: avvolgere un cavo od una catena  ad ampie spire  in modo che possano svolgersi liberamente e scorrere senza impedimenti. 

----------------------------------------
Literal Lemma: abbittare
Mariterm ID: V#577
Total Similarity: -0.33
Mariterm Gloss: legare un cavo o una cat

In [28]:
def calculate_weighted_average(scores):
    """
    Calculate the weighted average based on proportional contribution.
    """
    absolute_sum = sum(abs(score) for score in scores)
    
    if absolute_sum == 0:
        return 0
    
    weights = [abs(score) / absolute_sum for score in scores]
    weighted_average = sum(score * weight for score, weight in zip(scores, weights))
    
    return weighted_average

def compute_adjusted_score(total_similarity, gloss_similarity, relation_similarities):
    """
    Compute the adjusted score using total similarity, gloss similarity, and relation similarities.
    """
    # Start with the total similarity
    initial_score = total_similarity
    
    # Calculate weighted average of relation similarities
    scores = list(relation_similarities.values())
    relation_weighted_average = calculate_weighted_average(scores)
    
    # Adjusted score combines the total similarity and relation weighted average
    adjusted_score = initial_score + relation_weighted_average
    
    return adjusted_score

def extract_and_compute_scores(mari_term_file, ital_wn_file, candidates_csv):
    """
    Extract word meanings and compute the adjusted scores for entries with two or more corresponding entries in ItalWN.
    """
    candidate_lemmas = set(pd.read_csv(candidates_csv, delimiter=';')['Shared Lemma'].str.lower())
    mari_term_wms = extract_word_meanings(parse_xml(mari_term_file))
    ital_wn_wms = extract_word_meanings(parse_xml(ital_wn_file))
    
    mari_term_wms = [wm for wm in mari_term_wms if any(lemma.lower() in candidate_lemmas for lemma in wm['lemmas'])]
    ital_wn_wms = [wm for wm in ital_wn_wms if any(lemma.lower() in candidate_lemmas for lemma in wm['lemmas'])]
    
    results = []
    
    for mari_term_wm in mari_term_wms:
        mari_term_lemma = mari_term_wm['first_literal_lemma']
        
        # Extract 'sense' from a different part of the structure if 'variants' key does not exist
        mari_term_sense = None
        if 'variants' in mari_term_wm:
            for variant in mari_term_wm['variants']:
                if variant['lemma'].lower() == mari_term_lemma.lower():
                    mari_term_sense = variant['sense']
                    break
        else:
            # Handle cases where 'variants' is not present
            # Assuming 'sense' might be in a different key or format
            if 'sense' in mari_term_wm:
                mari_term_sense = mari_term_wm['sense']
        
        if mari_term_sense is None:
            continue  # Skip if no sense information is found
        
        mari_term_id = mari_term_wm['id']
        
        # Filter ItalWN word meanings based on both LEMMA and SENSE
        matching_ital_wn_wms = [
            wm for wm in ital_wn_wms
            if any(
                lemma.lower() == mari_term_lemma.lower() and
                sense == mari_term_sense
                for lemma, sense in zip(wm['lemmas'], wm['senses'])
            )
        ]
        
        if len(matching_ital_wn_wms) >= 2:
            comparison_scores = {}
            gloss_similarities = {}
            for wm in matching_ital_wn_wms:
                gloss_similarity = calculate_gloss_similarity(mari_term_wm['gloss'], wm['gloss'])
                gloss_similarities[wm['id']] = gloss_similarity
                
                relation_similarities = {}
                _, relation_weights, _, _, _, _, _, _, _, _ = calculate_relation_similarity(
                    mari_term_wm['relations'], wm['relations'], gloss_similarity
                )
                for key, (weighted_similarity, _, _) in relation_weights.items():
                    relation_similarities[key] = weighted_similarity
                
                # Compute total similarity based on provided example
                total_similarity = sum([
                    gloss_similarity,
                    sum(relation_similarities.values()),  # total weighted relation similarity
                    0,  # bonus
                    -0.33  # malus
                ])
                
                adjusted_score = compute_adjusted_score(total_similarity, gloss_similarity, relation_similarities)
                comparison_scores[wm['id']] = adjusted_score
            
            results.append({
                "literal_lemma": mari_term_lemma,
                "mari_term_id": mari_term_id,
                "ital_wn_entries": len(matching_ital_wn_wms),
                "ital_wn_comparison_scores": comparison_scores,
                "gloss_similarities": gloss_similarities,
                "details": {
                    "relation_similarities": relation_similarities
                }
            })
    # Return results and other data if needed
    return results, mari_term_wms

# Assuming results is a list of dictionaries and should be directly usable by print_results
def print_results(results):
    """
    Print the results for each literal lemma in MariTerm that corresponds to two or more literal lemmas in ItalWN.
    """
    if not results:
        print("No results found.")
        return
    
    for result in results:
        # Ensure result is a dictionary
        if isinstance(result, dict):
            print(f"\nLiteral Lemma: {result['literal_lemma']}, MariTerm ID: {result['mari_term_id']}")
            print(f" ItalWN Entries: {result['ital_wn_entries']}")
            
            for ital_id in result['ital_wn_comparison_scores']:
                score = result['ital_wn_comparison_scores'][ital_id]
                gloss_similarity = result['gloss_similarities'].get(ital_id, 0)
                print(f"  ItalWN ID: {ital_id} - Score: {score:.3f} (Gloss Similarity: {gloss_similarity:.3f})")
            
            average_score = calculate_weighted_average(list(result['ital_wn_comparison_scores'].values()))
            print(f" Adjusted Score (average): {average_score:.3f}")
        else:
            print("Unexpected data format in results")

# Extract and compute scores
results, _ = extract_and_compute_scores(mari_term_file, ital_wn_file, candidates_csv)

# Print results
print_results(results)


No results found.


In [29]:
def normalize_score(score, min_score, max_score, new_max=0.85):
    """Normalize a score to a range between 0 and new_max."""
    if min_score == max_score:
        return new_max  # Avoid division by zero
    return ((score - min_score) / (max_score - min_score)) * new_max

def print_formatted_results(mari_term_file, ital_wn_file, candidates_csv):
    """
    Print results in the specified format using the output from store_one_to_one.
    """
    # Call the store_one_to_one function from another cell
    unique_entries = store_one_to_one(mari_term_file, ital_wn_file, candidates_csv)

    # Check if there are any entries to process
    if not unique_entries:
        print("No valid entries to display.")
        return
    
    # Normalize Total Similarity
    total_similarities = [entry['Total Similarity'] for entry in unique_entries]
    min_similarity = min(total_similarities)
    max_similarity = max(total_similarities)
    
    # Sort results by Total Similarity in descending order
    unique_entries.sort(key=lambda x: x['Total Similarity'], reverse=True)
    
    # Print headers
    print(f"{'Literal Lemma':<25} {'MariTerm ID':<15} {'Total Similarity':<15} {'Normalized Score':<20} {'Multiple Entries in ItalWN':<30} {'MariTerm Gloss':<50}")
    print('-' * 150)
    
    # Print and store the results with normalization
    for entry in unique_entries:
        literal_lemma = entry.get('Literal Lemma', 'N/A')
        mari_term_id = entry.get('Mariterm ID', 'N/A')
        total_similarity = entry.get('Total Similarity', 0.00)
        normalized_score = normalize_score(total_similarity, min_similarity, max_similarity)
        gloss = entry.get('Mariterm Gloss', 'No Gloss Available')

        # Determine the multiple entries label
        multiple_entries = 'No (1)'  # As per your format, all should be No (1) in this case
        
        # Print the result
        print(f"{literal_lemma:<25} {mari_term_id:<15} {total_similarity:<15.2f} {normalized_score:<20.2f} {multiple_entries:<30} {gloss:<50}")

# Example usage
print_formatted_results(mari_term_file, ital_wn_file, candidates_csv)


Literal Lemma             MariTerm ID     Total Similarity Normalized Score     Multiple Entries in ItalWN     MariTerm Gloss                                    
------------------------------------------------------------------------------------------------------------------------------------------------------
nave militare             N#20            17.33           0.85                 No (1)                         nave impiegata per scopi militari e/o di guerra.  
imbarcazione              N#58            12.51           0.71                 No (1)                         termine generico per indicare barche di piccole e medie dimensioni  utilizzate per diversi scopi (lavoro  piacere  etc.).

oggetto                   N#645           8.80            0.59                 No (1)                         qualunque oggetto concreto.                       
inerente                  AG#127          5.77            0.50                 No (1)                                               

In [30]:

def save_results_to_csv(all_results, transcription_csv):
    """
    Save all results to a CSV file in the specified format, sorted by Total Similarity in descending order.
    """
    # Sort the results by Total Similarity in descending order
    all_results_sorted = sorted(all_results, key=lambda x: x.get('Total Similarity', 0), reverse=True)
    
    # Prepare data for DataFrame
    formatted_results = []

    for result in all_results_sorted:
        literal_lemma = result.get('Literal Lemma', 'N/A')
        mari_term_id = result.get('Mariterm ID', 'N/A')
        total_similarity = result.get('Total Similarity', 0.00)
        normalized_score = result.get('Normalized Score', 0.00)
        
        # Determine the multiple entries label
        multiple_entries = result.get('Multiple Entries in ItalWN', 'No (1)')
        gloss = result.get('Mariterm Gloss', 'No Gloss Available')
        
        formatted_results.append({
            'Literal Lemma': literal_lemma,
            'MariTerm ID': mari_term_id,
            'Total Similarity': f"{total_similarity:.2f}",
            'Normalized Score': f"{normalized_score:.2f}",
            'Multiple Entries in ItalWN': multiple_entries,
            'MariTerm Gloss': gloss
        })

    # Convert to DataFrame
    df = pd.DataFrame(formatted_results)
    
    # Save to CSV
    df.to_csv(transcription_csv, sep=';', index=False, float_format='%.2f')

def count_and_list_unique_lemmas(transcription_csv):
    """
    Count the number of unique Literal Lemmas and print a plain list of unique Lemmas.
    """
    # Load results from CSV
    df = pd.read_csv(transcription_csv, delimiter=';')
    # Get unique Literal Lemmas
    unique_lemmas = df['Literal Lemma'].dropna().unique()
    # Count unique Lemmas
    num_unique_lemmas = len(unique_lemmas)
    # Print the number of unique Lemmas
    print(f"Number of unique Literal Lemmas: {num_unique_lemmas}")
    # Print the list of unique Lemmas
    for lemma in unique_lemmas:
        print(lemma)


# Assuming `results` and `store_one_to_one` have been defined and used previously
one_to_one = store_one_to_one(mari_term_file, ital_wn_file, candidates_csv)
# Assuming `results` is the previously saved or additional results
all_results = one_to_one  # Replace `results` with actual data if necessary
save_results_to_csv(all_results, transcription_csv)
count_and_list_unique_lemmas(transcription_csv)


Number of unique Literal Lemmas: 1090
nave militare
imbarcazione
oggetto
inerente
bussola
galleggiante
porto
attività
unità_di_misura
lavoratore
attività_sportiva
prodotto
cardine
strumento nautico
fenomeno_atmosferico
roba
duello
bagaglio
persona
pesce
altezza
nulla_osta
circumnavigazione
fardello
ufficio
insieme
insenatura
rivestitura
remo
rimorchiatore
evento
fondo
scalo
viaggio
sottufficiale
dichiarazione
fenomeno_naturale
azimut
campionato
dazio
eliminatoria
ora
semifinale
alberatura
balla
import
informativa
poppavia
porto commerciale
porto di rifornimento
porto di velocità
porto industriale
porto petrolifero
sartiame
segnale
stazza
velame
aereo
albero dell'antenna
albero di fortuna
albero di maestra
ammattare
andatura al lasco
andatura di bolina
andatura in poppa
austro
binocolo
bolina
bordeggio
bovo
bussola da rilevamento
bussola di rotta
bussola giroscopica
bussola magnetica
calumare
campana
caravella
carroponte
cirro
cirrocumulo
cirrostrato
collettame
continentale
dogana
expor